# ESM2 foundation-model benchmark

Which ESM2 checkpoint up to 650M gives the best downstream accuracy, on
average across all datasets?

The representation is always mean-pooled ESM2 embeddings; only the checkpoint
changes (8M, 35M, 150M, 650M). The downstream model and its Optuna budget are
identical throughout, so any difference is attributable to the embedding.

Two levels of comparison, both a repeated-measures ANOVA followed by a Tukey
HSD on the metric values themselves:

1. **Within a dataset** — the random seed is the repeated measure, one test per
   dataset and training size.
2. **Across datasets** — each (family, dataset, train size) cell is the
   repeated measure, on the mean over seeds. This is what answers the headline
   question: whether one checkpoint is better *on average across datasets*,
   rather than on a single one.

## Configuration


In [28]:
import os

import numpy as np
import pandas as pd
import pingouin as pg

from model_comparison import rm_tukey_hsd

# ------------------------------------------------------------------
# The ESM2 checkpoints up to 650M, smallest first. Kept in the same order as
# ESM2_VARIANTS in simulate_experiments/functions/model_optimization.py.
# ------------------------------------------------------------------
VARIANT_LABELS = {
    "facebook/esm2_t6_8M_UR50D":    "ESM2 8M",
    "facebook/esm2_t12_35M_UR50D":  "ESM2 35M",
    "facebook/esm2_t30_150M_UR50D": "ESM2 150M",
    "facebook/esm2_t33_650M_UR50D": "ESM2 650M",
}

VARIANT_ORDER = ["ESM2 8M", "ESM2 35M", "ESM2 150M", "ESM2 650M"]

VARIANT_DIM = {"ESM2 8M": 320, "ESM2 35M": 480, "ESM2 150M": 640, "ESM2 650M": 1280}

FAMILIES = ["substitutions", "indels", "peptides"]

import os


def find_results_root(*candidates):
    """Return the first results folder that exists.

    The SLURM jobs write to simulate_experiments/simulation_output/; the
    simulation_outputs/ folder is only a copy (e.g. results moved off the
    cluster), so the jobs' own output is checked first and a stale copy can
    never shadow fresh results.
    """
    for c in candidates:
        if os.path.isdir(c):
            print(f"Reading results from {os.path.abspath(c)}")
            return c
    raise FileNotFoundError(
        "No benchmark results folder found. Looked in:\n  "
        + "\n  ".join(os.path.abspath(c) for c in candidates)
    )

SIM_ROOT = find_results_root(
    "../simulate_experiments/simulation_output/foundations",
    "../simulation_output/foundations",
    "../simulation_outputs/foundations",
)

# Tables below are written here.
os.makedirs("results", exist_ok=True)

# Whether a higher value of each metric is better.
METRIC_DIRECTION = {
    "Test MSE": "minimize",
    "Test RMSE": "minimize",
    "Test MAE": "minimize",
    "Test R2": "maximize",
    "Test Spearman Correlation": "maximize",
    "Top 20 Accuracy": "maximize",
}

metric_ls = list(METRIC_DIRECTION)

#: The metric the headline "which checkpoint is best" answer is based on.
PRIMARY_METRIC = "Test Spearman Correlation"


def anova_p(result):
    """Uncorrected p-value from pg.rm_anova, whichever pingouin version is installed.

    The column was renamed from "p-unc" to "p_unc" in pingouin 0.6.
    """
    for col in ("p_unc", "p-unc"):
        if col in result.columns:
            return float(result[col].iloc[0])
    raise KeyError(f"No p-value column in pg.rm_anova output: {list(result.columns)}")


Reading results from /Users/leoan/Documents/GitHub/SCARSE-repro/code/benchmark/simulate_experiments/simulation_output/foundations


## Read data


In [29]:
# ------------------------------------------------------------------
# Load every family's per-run results into one long table.
# ------------------------------------------------------------------
frames = []
for family in FAMILIES:
    path = os.path.join(SIM_ROOT, family, "simulation_summary_all.csv")
    if not os.path.exists(path):
        print(f"Missing (run the foundations_{family}.sh sweep and aggregate it): {path}")
        continue
    d = pd.read_csv(path)
    d["Family"] = family
    frames.append(d)

if not frames:
    raise FileNotFoundError(f"No foundation-benchmark results found under {SIM_ROOT}")

df = pd.concat(frames, ignore_index=True)

# Every run in this benchmark uses the ESM representation; only the checkpoint varies.
if set(df["Representation"].unique()) != {"esm"}:
    raise ValueError(
        "This benchmark expects only the 'esm' representation. Found: "
        f"{sorted(df['Representation'].unique())}"
    )

unknown = set(df["Foundation"].unique()) - set(VARIANT_LABELS)
if unknown:
    raise ValueError(f"Unexpected checkpoints in the results: {sorted(unknown)}")

df["Variant"] = df["Foundation"].map(VARIANT_LABELS)
df["method"] = df["Variant"]
df["cv_cycle"] = df["Seed"]

print(f"{len(df)} runs")
print(f"Checkpoints: {[v for v in VARIANT_ORDER if v in set(df['Variant'])]}")
for family in df["Family"].unique():
    d = df[df["Family"] == family]
    print(f"  {family:14s} {d['Dataset'].nunique():2d} datasets  "
          f"train sizes {sorted(d['Train Size'].unique())}  "
          f"{d['Seed'].nunique()} seeds")


5520 runs
Checkpoints: ['ESM2 8M', 'ESM2 35M', 'ESM2 150M', 'ESM2 650M']
  substitutions  10 datasets  train sizes [np.int64(20), np.int64(50), np.int64(75), np.int64(100), np.int64(200), np.int64(350), np.int64(500)]  10 seeds
  indels         10 datasets  train sizes [np.int64(20), np.int64(40), np.int64(60), np.int64(80)]  10 seeds
  peptides        4 datasets  train sizes [np.int64(20), np.int64(50), np.int64(75), np.int64(100), np.int64(200), np.int64(350), np.int64(500)]  10 seeds


## Check the design is complete


In [30]:
# ------------------------------------------------------------------
# The repeated-measures tests need every checkpoint present for every seed.
# ------------------------------------------------------------------
cells = []
incomplete = []

for (family, dataset, size), d in df.groupby(["Family", "Dataset", "Train Size"]):
    counts = d.pivot_table(index="cv_cycle", columns="Variant",
                           values="Test MSE", aggfunc="size")
    if counts.shape[1] < len(VARIANT_LABELS) or counts.isna().any().any():
        incomplete.append((family, dataset, size,
                           f"{counts.shape[1]}/{len(VARIANT_LABELS)} checkpoints, "
                           f"{int(counts.isna().sum().sum())} missing seed cells"))
    else:
        cells.append((family, dataset, size))

print(f"{len(cells)} complete (family, dataset, train size) cells")
if incomplete:
    print(f"\n{len(incomplete)} incomplete cells will be skipped:")
    for row in incomplete[:15]:
        print(f"  {row[0]} / {row[1]} / train size {row[2]}: {row[3]}")
    if len(incomplete) > 15:
        print(f"  ... and {len(incomplete) - 15} more")
else:
    print("Design is complete.")

skip = {(f, d, s) for f, d, s, _ in incomplete}


138 complete (family, dataset, train size) cells
Design is complete.


## Level 1 — within each dataset (seeds as repeated measure)


In [31]:
# ------------------------------------------------------------------
# Level 1 — within each dataset, with the random seed as the repeated measure.
# Six pairwise comparisons per (family, dataset, train size, metric).
# ------------------------------------------------------------------
rows = []
nan_cells = []   # cells where a metric was undefined for some runs

for metric in metric_ls:
    for family, dataset, size in cells:
        d = df[(df["Family"] == family) &
               (df["Dataset"] == dataset) &
               (df["Train Size"] == size)]

        # A metric can be undefined for a run (Spearman is NaN when a model
        # predicts the same value for every test sequence). The repeated-measures
        # ANOVA only uses seeds that are complete across all groups; do that step
        # here, so it is reported and the Tukey means use the same seeds.
        wide = d.pivot_table(index="cv_cycle", columns="Variant",
                                   values=metric, dropna=False)
        undefined = [c for c in wide.columns if wide[c].isna().all()]
        wide = wide.drop(columns=undefined).dropna()
        n_seeds = d["cv_cycle"].nunique()
        if wide.shape[1] < 2 or wide.shape[0] < 2:
            nan_cells.append((metric, (family, dataset, size), "not tested", undefined,
                wide.shape[0] if wide.shape[1] else 0, n_seeds))
            continue
        if undefined or wide.shape[0] < n_seeds:
            nan_cells.append((metric, (family, dataset, size), "reduced", undefined, wide.shape[0], n_seeds))
        d = d[d["Variant"].isin(wide.columns)
            & d["cv_cycle"].isin(wide.index)]

        results = rm_tukey_hsd(d, metric, "Variant")[0]
        for _, r in results.iterrows():
            rows.append({
                "Family": family,
                "Dataset": dataset,
                "Train Size": size,
                "Metric": metric,
                "group1": r["group1"],
                "group2": r["group2"],
                "p-adj": r["p-adj"],
                "effect size": r["meandiff"],
            })

within_dataset = pd.DataFrame(rows)

if nan_cells:
    n_skip = sum(1 for r in nan_cells if r[2] == "not tested")
    print(f"Undefined (NaN) metric values in {len(nan_cells)} cells "
          f"({n_skip} could not be tested, {len(nan_cells) - n_skip} tested on fewer seeds).")
    print("A run's metric is NaN when it cannot be computed - Spearman is undefined")
    print("when a model predicts the same value for every test sequence.\n")
    for metric, where, status, undefined, kept, total in nan_cells:
        what = (f"undefined for {', '.join(undefined)}; " if undefined else "")
        print(f"  {metric:26s} {where[0]}/{where[1]}  train size {where[2]}: {status} - {what}{kept}/{total} complete seeds")
    print()
within_dataset.to_csv("results/foundations_within_dataset.csv", index=False)
print(f"Wrote results/foundations_within_dataset.csv ({len(within_dataset)} comparisons)")
within_dataset.head()


/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/parametric.py:575: RuntimeWarning: invalid value encountered in scalar divide
  fval = ms_with / ms_reswith
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/parametric.py:585: RuntimeWarning: invalid value encountered in scalar divide
  ef = ss_with / (ss_with + ss_resall)
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/pingouin/distribution.py:742: RuntimeWarning: invalid value encountered in scalar divide
  eps = np.min([num / den, 1])
/Users/leoan/Documents/GitHub/SCARSE-repro/code/benchmark/simulate_experiments/statistics/model_comparison.py:104: RuntimeWarning: invalid value encountered in scalar divide
  studentized_range = np.abs(mean_diff) / tukey_se
/Users/leoan/Documents/GitHub/SCARSE-repro/.venv/lib/python3.12/site-packages/scipy/optimize/_optimize.py:2427: OptimizeWarning: 
NaN result encountered.
  _endprint(x, flag, fva

Undefined (NaN) metric values in 21 cells (0 could not be tested, 21 tested on fewer seeds).
A run's metric is NaN when it cannot be computed - Spearman is undefined
when a model predicts the same value for every test sequence.

  Test Spearman Correlation  indels/POLG_PESV_Tsuboyama_2023_2MXD_indels  train size 20: reduced - 9/10 complete seeds
  Test Spearman Correlation  peptides/EC_all_40  train size 20: reduced - 9/10 complete seeds
  Test Spearman Correlation  peptides/PA_all_40  train size 20: reduced - 6/10 complete seeds
  Test Spearman Correlation  peptides/PA_all_40  train size 75: reduced - 9/10 complete seeds
  Test Spearman Correlation  peptides/SA_all_40  train size 20: reduced - 6/10 complete seeds
  Test Spearman Correlation  peptides/SA_all_40  train size 50: reduced - 9/10 complete seeds
  Test Spearman Correlation  peptides/hemopi2_all  train size 20: reduced - 5/10 complete seeds
  Test Spearman Correlation  peptides/hemopi2_all  train size 50: reduced - 9/10 compl

,Family,Dataset,Train Size,Metric,group1,group2,p-adj,effect size
0,indels,MAFG_MOUSE_Tsuboyama_2023_1K1V_indels,20,Test MSE,ESM2 150M,ESM2 35M,0.900000,-0.013774
1,indels,MAFG_MOUSE_Tsuboyama_2023_1K1V_indels,20,Test MSE,ESM2 150M,ESM2 650M,0.482030,-0.081605
2,indels,MAFG_MOUSE_Tsuboyama_2023_1K1V_indels,20,Test MSE,ESM2 150M,ESM2 8M,0.506109,-0.079198
3,indels,MAFG_MOUSE_Tsuboyama_2023_1K1V_indels,20,Test MSE,ESM2 35M,ESM2 650M,0.617050,-0.067831
4,indels,MAFG_MOUSE_Tsuboyama_2023_1K1V_indels,20,Test MSE,ESM2 35M,ESM2 8M,0.640541,-0.065424


## Mean score per checkpoint within every cell

In [32]:
# ------------------------------------------------------------------
# Mean over seeds of every metric, per (family, dataset, train size) cell and
# checkpoint. These cell means are the observations for level 2.
# ------------------------------------------------------------------
complete = set(cells)
in_design = [(f, d, s) in complete for f, d, s in
             zip(df["Family"], df["Dataset"], df["Train Size"])]
runs = df[in_design].copy()

cell_means = (
    runs.groupby(["Family", "Dataset", "Train Size", "Variant"], as_index=False)[metric_ls]
        .mean()
        .melt(id_vars=["Family", "Dataset", "Train Size", "Variant"],
              var_name="Metric", value_name="value")
)

cell_means.to_csv("results/foundations_cell_means.csv", index=False)
print(f"Wrote results/foundations_cell_means.csv ({len(cell_means)} rows)")

print(f"\nMean {PRIMARY_METRIC} per checkpoint over all cells:")
print(
    cell_means[cell_means["Metric"] == PRIMARY_METRIC]
    .groupby("Variant")["value"].agg(["mean", "std", "size"])
    .reindex(VARIANT_ORDER).round(4).to_string()
)

Wrote results/foundations_cell_means.csv (3312 rows)

Mean Test Spearman Correlation per checkpoint over all cells:
             mean     std  size
Variant                        
ESM2 8M    0.5875  0.2457   138
ESM2 35M   0.5998  0.2569   138
ESM2 150M  0.6159  0.2618   138
ESM2 650M  0.6241  0.2634   138


## Level 2 — across datasets (cells as repeated measure)

In [33]:
# ------------------------------------------------------------------
# Level 2 — across datasets, with each (family, dataset, train size) cell as
# the repeated measure: a repeated-measures ANOVA over the cells, then a Tukey
# HSD for every pair of checkpoints. Same test as level 1, one level up.
# ------------------------------------------------------------------
def across_datasets(frame, metric, label):
    """rm-ANOVA + Tukey HSD over the cells in `frame` (long: one row per cell x checkpoint)."""
    pivot = frame.pivot_table(index=["Family", "Dataset", "Train Size"],
                              columns="Variant", values="value")
    pivot = pivot[[v for v in VARIANT_ORDER if v in pivot.columns]].dropna()

    if pivot.shape[0] < 3 or pivot.shape[1] < 2:
        return None, None

    long = (pivot.reset_index()
                 .melt(id_vars=["Family", "Dataset", "Train Size"],
                       var_name="Variant", value_name="value"))
    long["cv_cycle"] = (long["Family"] + " | " + long["Dataset"] + " | "
                        + long["Train Size"].astype(str))

    higher_is_better = METRIC_DIRECTION[metric] == "maximize"
    means = pivot.mean()

    # If every checkpoint scores exactly the same in every cell there is nothing
    # to test: the ANOVA's F is 0/0, and pingouin then drops the F and p columns
    # altogether. Report that case instead of failing on it.
    spread = (pivot.max(axis=1) - pivot.min(axis=1)).abs().max()
    if not np.isfinite(spread) or spread == 0:
        print(f"  {label} / {metric}: all checkpoints identical in all "
              f"{pivot.shape[0]} cells (value {float(pivot.iloc[0].iloc[0]):g}) - "
              "no difference to test, reported as NaN")
        p_anova = np.nan
        best = None
        tukey = pd.DataFrame(
            [{"group1": a, "group2": b, "meandiff": 0.0, "p-adj": np.nan}
             for i, a in enumerate(pivot.columns) for b in pivot.columns[i + 1:]]
        )
    else:
        p_anova = anova_p(pg.rm_anova(data=long, dv="value", within="Variant",
                                      subject="cv_cycle", detailed=True))
        tukey = rm_tukey_hsd(long, "value", "Variant")[0]
        best = means.idxmax() if higher_is_better else means.idxmin()

    summary = pd.DataFrame({
        "Scope": label,
        "Metric": metric,
        "Variant": means.index,
        "mean value": means.values,
        "std across cells": pivot.std().values,
        "n cells": pivot.shape[0],
        "rm-ANOVA p": p_anova,
        "best": means.index == best,
    })

    pairs = pd.DataFrame({
        "Scope": label,
        "Metric": metric,
        "group1": tukey["group1"].values,
        "group2": tukey["group2"].values,
        "mean diff": tukey["meandiff"].astype(float).values,
        "Tukey p-adj": tukey["p-adj"].astype(float).values,
        "n cells": pivot.shape[0],
    })
    return summary, pairs


summaries, pairwise = [], []

for metric in metric_ls:
    sub = cell_means[cell_means["Metric"] == metric]

    s, p = across_datasets(sub, metric, "all families")
    if s is not None:
        summaries.append(s)
        pairwise.append(p)

    for family in FAMILIES:
        s, p = across_datasets(sub[sub["Family"] == family], metric, family)
        if s is not None:
            summaries.append(s)
            pairwise.append(p)

across_summary = pd.concat(summaries, ignore_index=True)
across_pairwise = pd.concat(pairwise, ignore_index=True)

across_summary.to_csv("results/foundations_across_datasets.csv", index=False)
across_pairwise.to_csv("results/foundations_across_datasets_pairwise.csv", index=False)
print(f"Wrote results/foundations_across_datasets.csv ({len(across_summary)} rows)")
print(f"Wrote results/foundations_across_datasets_pairwise.csv ({len(across_pairwise)} rows)")

across_summary[across_summary["Metric"] == PRIMARY_METRIC].round(4)

  indels / Top 20 Accuracy: all checkpoints identical in all 40 cells (value 100) - no difference to test, reported as NaN
Wrote results/foundations_across_datasets.csv (96 rows)
Wrote results/foundations_across_datasets_pairwise.csv (144 rows)


,Scope,Metric,Variant,mean value,std across cells,n cells,rm-ANOVA p,best
64,all families,Test Spearman Correlation,ESM2 8M,0.5875,0.2457,138,0.0000,False
65,all families,Test Spearman Correlation,ESM2 35M,0.5998,0.2569,138,0.0000,False
66,all families,Test Spearman Correlation,ESM2 150M,0.6159,0.2618,138,0.0000,False
67,all families,Test Spearman Correlation,ESM2 650M,0.6241,0.2634,138,0.0000,True
68,substitutions,Test Spearman Correlation,ESM2 8M,0.6133,0.2359,70,0.0000,False
69,substitutions,Test Spearman Correlation,ESM2 35M,0.6340,0.2342,70,0.0000,False
70,substitutions,Test Spearman Correlation,ESM2 150M,0.6548,0.2397,70,0.0000,False
71,substitutions,Test Spearman Correlation,ESM2 650M,0.6593,0.2465,70,0.0000,True
72,indels,Test Spearman Correlation,ESM2 8M,0.7510,0.0987,40,0.0000,False
73,indels,Test Spearman Correlation,ESM2 35M,0.7729,0.0973,40,0.0000,False


## Which checkpoint is best on average?


In [34]:
# ------------------------------------------------------------------
# The answer: which checkpoint is best on average across datasets.
# ------------------------------------------------------------------
overall = across_summary[across_summary["Scope"] == "all families"].copy()

table = (
    overall
    .pivot_table(index="Variant", columns="Metric", values="mean value")
    .reindex(VARIANT_ORDER)[[m for m in metric_ls if m in set(overall["Metric"])]]
)
table.insert(0, "embedding dim", [VARIANT_DIM[v] for v in table.index])

print("Mean over all (dataset, train size) cells:\n")
print(table.round(4).to_string())

print("\nBest checkpoint per metric (higher is better except for the error metrics):")
for metric in metric_ls:
    o = overall[overall["Metric"] == metric]
    if o.empty:
        continue
    if not o["best"].any():
        print(f"  {metric:28s} no difference: all checkpoints identical")
        continue
    w = o[o["best"]].iloc[0]
    print(f"  {metric:28s} {w['Variant']:10s} "
          f"(mean {w['mean value']:.4f}, rm-ANOVA p = {w['rm-ANOVA p']:.2g})")

# Is the best checkpoint actually separable from the others?
best = overall[(overall["Metric"] == PRIMARY_METRIC) & overall["best"]]["Variant"].iloc[0]
print(f"\nTukey HSD against {best}, {PRIMARY_METRIC}, across all cells:")
pw = across_pairwise[(across_pairwise["Scope"] == "all families") &
                     (across_pairwise["Metric"] == PRIMARY_METRIC)]
for _, r in pw[(pw["group1"] == best) | (pw["group2"] == best)].iterrows():
    other = r["group2"] if r["group1"] == best else r["group1"]
    diff = r["mean diff"] if r["group1"] == best else -r["mean diff"]
    verdict = "significant" if r["Tukey p-adj"] < 0.05 else "not significant"
    print(f"  vs {other:10s} {best} - {other} = {diff:+.4f}   "
          f"p-adj = {r['Tukey p-adj']:.3g}  ({verdict})")

table.round(4).to_csv("results/foundations_mean_table.csv")
print("\nWrote results/foundations_mean_table.csv")
table.round(4)

Mean over all (dataset, train size) cells:

Metric     embedding dim  Test MSE  Test RMSE  Test MAE  Test R2  Test Spearman Correlation  Top 20 Accuracy
Variant                                                                                                     
ESM2 8M              320    0.5313     0.6864    0.5247   0.3813                     0.5875          43.4312
ESM2 35M             480    0.4988     0.6603    0.5031   0.4123                     0.5998          43.6630
ESM2 150M            640    0.4730     0.6354    0.4830   0.4402                     0.6159          45.2609
ESM2 650M           1280    0.4600     0.6280    0.4734   0.4470                     0.6241          46.2319

Best checkpoint per metric (higher is better except for the error metrics):
  Test MSE                     ESM2 650M  (mean 0.4600, rm-ANOVA p = 5.6e-16)
  Test RMSE                    ESM2 650M  (mean 0.6280, rm-ANOVA p = 1.5e-21)
  Test MAE                     ESM2 650M  (mean 0.4734, rm-ANOVA p = 

Metric,embedding dim,Test MSE,Test RMSE,Test MAE,Test R2,Test Spearman Correlation,Top 20 Accuracy
Variant,,,,,,,
ESM2 8M,320,0.5313,0.6864,0.5247,0.3813,0.5875,43.4312
ESM2 35M,480,0.4988,0.6603,0.5031,0.4123,0.5998,43.6630
ESM2 150M,640,0.4730,0.6354,0.4830,0.4402,0.6159,45.2609
ESM2 650M,1280,0.4600,0.6280,0.4734,0.4470,0.6241,46.2319
